## Split Baseline vs Current
## Baseline = 2016–2025 (history used to build climate normals). Current = 2026 (the period we're scoring for anomalies).

In [0]:
from pyspark.sql.functions import year

df_silver = spark.table("climate_project.silver_observations")

df_baseline = df_silver.filter(year(df_silver.DATE) <= 2025)
df_current = df_silver.filter(year(df_silver.DATE) == 2026)

print("Baseline rows:", df_baseline.count())
print("Current rows:", df_current.count())

Baseline rows: 198120406
Current rows: 11838351


%md
### Per-Day Raw Stats
### For each station/element/day-of-year, sum the raw values and squared values (not the average) . These raw sums are what let us correctly combine multiple days together later in the rolling window.

In [0]:
from pyspark.sql.functions import dayofyear, sum as _sum, count as _count, expr

df_daily_stats = df_baseline.groupBy(
    "ID", "ELEMENT",
    dayofyear("DATE").alias("doy")
).agg(
    _sum("DATA_VALUE").alias("sum_value"),
    _sum(expr("DATA_VALUE * DATA_VALUE")).alias("sum_sq_value"),
    _count("DATA_VALUE").alias("day_count")
)

df_daily_stats.show(10)
print(df_daily_stats.count())

+-----------+-------+---+------------------+------------------+---------+
|         ID|ELEMENT|doy|         sum_value|      sum_sq_value|day_count|
+-----------+-------+---+------------------+------------------+---------+
|ASN00030158|   PRCP|  1|              41.4|             941.0|       10|
|ASN00021135|   PRCP|  1|               0.0|               0.0|        3|
|ASN00017119|   PRCP|  1|               0.0|               0.0|       10|
|ASN00039083|   PRCP|  1|               6.8|22.000000000000004|        9|
|ASN00041533|   PRCP|  1|              35.0|            1157.0|       10|
|ASN00070217|   TMIN|  1|             108.8|           1213.66|       10|
|BN000005319|   TMIN|  1|             133.3|1993.4499999999996|        9|
|CA001021261|   PRCP|  1|              53.8|1384.1000000000001|        9|
|CA001062544|   TMAX|  1|              66.5|            532.75|        9|
|CA001106CL2|   PRCP|  1|20.799999999999997|307.43999999999994|        9|
+-----------+-------+---+-------------

## Roll Into a ±7-Day Window
### Instead of comparing to just one exact day, pool each day with its 7 neighbors on either side (up to 15 days, across all years) by summing the raw sums over a rolling window. Fixes sparse baselines caused by stations with irregular reporting schedules.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import sum as _sum, col, sqrt

window_days = 7

day_window = Window.partitionBy("ID", "ELEMENT").orderBy("doy").rowsBetween(-window_days, window_days)

df_rolled = df_daily_stats.withColumn("rolling_sum_value", _sum("sum_value").over(day_window)) \
                           .withColumn("rolling_sum_sq", _sum("sum_sq_value").over(day_window)) \
                           .withColumn("rolling_count", _sum("day_count").over(day_window))

df_rolled.select("ID", "ELEMENT", "doy", "rolling_sum_value", "rolling_sum_sq", "rolling_count", ).show(10)

+-----------+-------+---+------------------+------------------+-------------+
|         ID|ELEMENT|doy| rolling_sum_value|    rolling_sum_sq|rolling_count|
+-----------+-------+---+------------------+------------------+-------------+
|AG000060390|   PRCP|  1|              82.7|            744.07|           29|
|AG000060390|   PRCP|  2|              85.2|            748.32|           35|
|AG000060390|   PRCP|  3|              95.8| 783.3800000000001|           41|
|AG000060390|   PRCP|  4|             171.3|           2327.03|           49|
|AG000060390|   PRCP|  5|222.10000000000002|2846.8100000000004|           57|
|AG000060390|   PRCP|  6|243.10000000000002|3020.4300000000003|           66|
|AG000060390|   PRCP|  7|267.20000000000005|3339.4400000000005|           73|
|AG000060390|   PRCP|  8|             281.1|3445.4500000000007|           78|
|AG000060390|   PRCP|  9|             340.1| 5331.070000000001|           81|
|AG000060390|   PRCP| 10|             380.2|           7541.06| 

## Compute Baseline & Filter Thin Data
### Derive mean/variance/stddev from the rolled sums, then drop any baseline built from fewer than 30 pooled observations becasue they have too little history to trust.

In [0]:
df_baseline_final = df_rolled.withColumn("baseline_mean", col("rolling_sum_value") / col("rolling_count")) \
                              .withColumn("baseline_variance", col("rolling_sum_sq") / col("rolling_count") - col("baseline_mean") ** 2) \
                              .withColumn("baseline_stddev", sqrt(col("baseline_variance")))
                              
df_baseline_final = df_baseline_final.filter(col("rolling_count") >= 30)

df_baseline_final.select("ID", "ELEMENT", "doy", "rolling_count", "baseline_mean", "baseline_stddev").show(10)

+-----------+-------+---+-------------+------------------+------------------+
|         ID|ELEMENT|doy|rolling_count|     baseline_mean|   baseline_stddev|
+-----------+-------+---+-------------+------------------+------------------+
|AG000060390|   PRCP|  2|           35| 2.434285714285714| 3.931262454962263|
|AG000060390|   PRCP|  3|           41|2.3365853658536584|3.6942114309242253|
|AG000060390|   PRCP|  4|           49| 3.495918367346939| 5.938767795773095|
|AG000060390|   PRCP|  5|           57| 3.896491228070176| 5.895879170852425|
|AG000060390|   PRCP|  6|           66|3.6833333333333336|5.6742529433086135|
|AG000060390|   PRCP|  7|           73|3.6602739726027402| 5.687543219189152|
|AG000060390|   PRCP|  8|           78| 3.603846153846154| 5.584328858228551|
|AG000060390|   PRCP|  9|           81|4.1987654320987655|  6.94161709247623|
|AG000060390|   PRCP| 10|           82| 4.636585365853659|  8.39441614923963|
|AG000060390|   PRCP| 11|           84| 5.213095238095238| 9.130

## Saving the cleaned climate normals as a Delta table.

In [0]:
df_baseline_final.write.format("delta").mode("overwrite").saveAsTable("climate_project.gold_baseline")

## Adding a `doy` column to the 2026 data so it can be matched against the baseline, which is keyed by day-of-year.


In [0]:
df_current_with_doy = df_current.withColumn("doy", dayofyear("DATE"))

## Joining Current to Baseline for matching each 2026 reading to its station/element/day-of-year baseline.

In [0]:
df_gold_baseline = spark.table("climate_project.gold_baseline")

df_scored = df_current_with_doy.join(
    df_gold_baseline,
    on=["ID", "ELEMENT", "doy"],
    how="inner"
)

%md
## Compute Z-Score
### How many standard deviations each reading falls from its historical baseline. `try_divide` returns null instead of erroring when `baseline_stddev` is 0 (prevention from scenarion where no historical variation exits to compare against).

In [0]:
from pyspark.sql.functions import col, try_divide

df_scored = df_scored.withColumn(
    "z_score",
    try_divide(col("DATA_VALUE") - col("baseline_mean"), col("baseline_stddev"))
)

## Previewing a sample station's scored rows to confirm the numbers look reasonable before filtering.

In [0]:
df_scored.select("ID", "ELEMENT", "DATE", "DATA_VALUE", "baseline_mean", "baseline_stddev", "z_score").show(10)

+-----------+-------+----------+----------+-----------------+------------------+--------------------+
|         ID|ELEMENT|      DATE|DATA_VALUE|    baseline_mean|   baseline_stddev|             z_score|
+-----------+-------+----------+----------+-----------------+------------------+--------------------+
|USC00406012|   PRCP|2026-01-01|       0.0|5.296153846153846|14.743468282665798|-0.35922035064033364|
|USC00406012|   PRCP|2026-01-02|       0.0|6.010227272727272|15.668229578678165|-0.38359326065187255|
|USC00406012|   PRCP|2026-01-03|       2.0|5.654166666666666|15.066967986478089|-0.24252833549166047|
|USC00406012|   PRCP|2026-01-04|       0.0|              5.6|14.742255432534579| -0.3798604647455368|
|USC00406012|   PRCP|2026-01-05|       0.3|5.464601769911504|14.328546421061974|-0.36044143056408684|
|USC00406012|   PRCP|2026-01-06|       0.0|5.276229508196722|13.870005692664266| -0.3804057204523915|
|USC00406012|   PRCP|2026-01-07|       0.3|4.903787878787879|13.399193809438067|-0

## Keeping only readings more than 3 standard deviations from their baseline which is the conventional threshold for being kind of statistically rare.

In [0]:
from pyspark.sql.functions import abs as _abs

df_anomalies = df_scored.filter(_abs(col("z_score")) > 3)
df_anomalies.count()

203848

## Checking which weather element (TMAX/TMIN/PRCP) makes up the flagged anomalies. PRCP dominates due to its skewed, zero-inflated distribution.


In [0]:
df_anomalies.groupBy("ELEMENT").count().orderBy("count", ascending=False).show()

+-------+------+
|ELEMENT| count|
+-------+------+
|   PRCP|163661|
|   TMAX| 21425|
|   TMIN| 18762|
+-------+------+



## Joining station metadata onto the anomalies so each row shows a real name and coordinates instead of just a bare station ID.


In [0]:
df_stations = spark.table("climate_project.silver_stations")

df_anomalies_labeled = df_anomalies.join(
    df_stations,
    on="ID",
    how="left"
)

## Sorting by absolute z-score to surface the single most unusual readings in the dataset.

In [0]:
df_anomalies_labeled.select(
    "NAME", "STATE", "ELEMENT", "DATE", "DATA_VALUE", "baseline_mean", "baseline_stddev", "rolling_count", "z_score"
).orderBy(_abs(col("z_score")), ascending=False).show(10, truncate=False)

+------------------------------+-----+-------+----------+----------+--------------------+--------------------+-------------+------------------+
|NAME                          |STATE|ELEMENT|DATE      |DATA_VALUE|baseline_mean       |baseline_stddev     |rolling_count|z_score           |
+------------------------------+-----+-------+----------+----------+--------------------+--------------------+-------------+------------------+
|YUDNAPINNA                    |     |PRCP   |2026-02-22|122.0     |0.008888888888888889|0.08731947868617111 |135          |1397.0664157255326|
|EDEOWIE                       |     |PRCP   |2026-02-23|74.7      |0.007333333333333334|0.0566823507706666  |150          |1317.7411601870347|
|MARREE (FARINA)               |     |PRCP   |2026-03-02|37.6      |0.002666666666666667|0.03255081497528988 |150          |1155.035084739795 |
|ROXBY DOWNS (OLYMPIC DAM AEROD|     |PRCP   |2026-02-23|41.4      |0.004054054054054054|0.03653650225566426 |148          |1133.0024329

%md
## During development, baselines built from fewer than 30 pooled historical observations turned out to be unreliable .Thin baselines produced near-zero standard deviations, which inflated some z-scores into the thousands and even produced physically impossible values (e.g., a tropical station's baseline showing sub-zero temperatures). `gold_baseline` now filters these out at the source; this check confirms none slipped through into the final result.

In [0]:
df_anomalies_labeled.filter(col("rolling_count") < 30).count()

0

## Saved the final, cleaned and labeled anomaly results

In [0]:
df_anomalies_labeled.write.format("delta").mode("overwrite").saveAsTable("climate_project.gold_anomalies")